# Tutoriel K-ABENA — CNN (niveau 1 : notebook)
CNN compact sur MNIST/CIFAR-léger. **PyTorch et TF/Keras**. Le motif est identique au MLP :
la seule différence est le modèle — c'est le point du design K-ABENA (agnostique à l'architecture).
⚠ Exécuter de préférence avec GPU pour CIFAR ; MNIST passe en CPU.

## Section A — PyTorch

In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms
from kabena.integrations.torch import KabenaTorch

tfm = transforms.ToTensor()
train = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(train, batch_size=512, shuffle=True)

model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(16*7*7, 10))
opt = torch.optim.SGD(model.parameters(), lr=0.1)     # SGD recommandé (Limitation L1)
crit = nn.CrossEntropyLoss(reduction="none")

kb = KabenaTorch(seed=0)                               # <= LIGNE 1
for epoch in range(3):
    for xb, yb in loader:
        losses = crit(model(xb), yb)
        loss = kb.reduce(losses, y=yb)                 # <= LIGNE 2 (par batch)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}: gain courant = {kb.last_gain_*100:.1f}%")

## Section B — TensorFlow/Keras
Même callback `KabenaKeras` que le MLP — remplacez simplement le modèle par un `Conv2D` stack ;
voir `niveau2_script_tensorflow.py` pour la version complète prête à lancer.

**Pourquoi SGD et pas Adam ici ?** Le preprint (Limitation L1) mesure qu'en déséquilibre
extrême Adam+K-ABENA sous-performe (AUC 0,806 vs 0,9991 pour SGD) — en régime standard les
deux sont neutres, mais SGD est le choix couvert par le Théorème 1.